## Step 1: Setup
We install the libraries, check the GPU and set the pilot size. Change the numbers in the config cell only if you are short of time (smaller `N_TRAIN` and `MAX_STEPS` = faster).

In [1]:
%pip install -q transformers "datasets<4" jiwer soundfile statsmodels huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
import os, re, io, json, random, time, warnings
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")

ON_KAGGLE = os.path.exists("/kaggle")
WORK = "/kaggle/working/pilot" if ON_KAGGLE else "/content/pilot"
os.makedirs(WORK, exist_ok=True)

SEED = 42
MODEL_NAME = "openai/whisper-small"          
N_TEST, N_TRAIN = 100, 1500                  
MAX_STEPS, BATCH, GRAD_ACCUM = 400, 8, 2     
LR, WARMUP = 1e-5, 40
FREEZE_ENCODER = False                       
N_VAANI = 200                                
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NONE - switch the runtime to GPU, CPU will be far too slow")
print("work folder:", WORK)

GPU: Tesla T4
work folder: /kaggle/working/pilot


## Step 2: Metric helpers (WER) and a self-test
**WER** = (substitutions + deletions + insertions) / words in the reference. Everything is lower-cased and stripped of punctuation first.

Two details worth knowing:
- We remove punctuation by Unicode category, **not** with the regex `\w`, because `\w` deletes Devanagari vowel signs and would corrupt Hindi text.
- `multiref_errors` is a pilot version of the VAANI "minimum-error" multi-reference score (keep the reference with the fewest errors).

**You should see:** `helper self-test passed`. If an assertion fails, stop and tell your assistant.

In [ ]:
import re, unicodedata
import numpy as np
import pandas as pd
import jiwer

def normalise(s: str) -> str:
    """Lower-case, drop punctuation/symbols, keep letters AND Devanagari vowel signs, squeeze spaces.
    NOTE: we do NOT use the regex \\w here, because Python's \\w drops Devanagari vowel signs (matras)."""
    s = unicodedata.normalize("NFC", s).lower()
    s = "".join(" " if unicodedata.category(c)[0] in "PS" else c for c in s)
    return " ".join(s.split())

def strip_brackets(s: str) -> str:
    """Remove annotation chunks from VAANI references: <noise>, <PAUSE>, [breathing], {alternate form}, (..).
    Without this, tag words such as 'noise' and 'pause' would be counted as real words."""
    return re.sub(r"[\[\(\{<][^\]\)\}>]*[\]\)\}>]", " ", s)

def word_errors(ref: str, hyp: str):
    """Return (substitutions, deletions, insertions, ref_words) for ONE pair of already-normalised strings."""
    r, h = ref.split(), hyp.split()
    if not r:
        raise ValueError("empty reference")
    if not h:                       
        return 0, len(r), 0, len(r)
    o = jiwer.process_words(ref, hyp)
    return o.substitutions, o.deletions, o.insertions, len(r)

def score_pairs(refs, hyps):
    """Per-utterance error table + corpus WER. Corpus WER = total errors / total reference words."""
    rows = []
    for i, (r, h) in enumerate(zip(refs, hyps)):
        rn, hn = normalise(r), normalise(h)
        if not rn:
            continue
        s, d, ins, n = word_errors(rn, hn)
        rows.append(dict(idx=i, ref=rn, hyp=hn, S=s, D=d, I=ins, errors=s + d + ins, ref_words=n))
    df = pd.DataFrame(rows)
    wer = df["errors"].sum() / df["ref_words"].sum()
    return df, float(wer)

def multiref_errors(refs, hyp):
    """Pilot version of minimum-error multi-reference scoring (Approach 3 in the VAANI Benchmark paper):
    score the hypothesis against every reference and keep the reference with the FEWEST errors.
    Returns (errors, ref_words). The OFFICIAL toolkit is used for final numbers."""
    hn = normalise(strip_brackets(hyp))
    best = None
    for r in refs:
        rn = normalise(strip_brackets(r))
        if not rn:
            continue
        s, d, i, n = word_errors(rn, hn)
        e = s + d + i
        if best is None or e < best[0]:
            best = (e, n)
    return best

def latin_share(texts):
    """Share of letters written in Latin script. Used to see the Latin-vs-Devanagari mismatch between references and outputs."""
    lat = dev = 0
    for t in texts:
        for c in t:
            if c.isalpha():
                name = unicodedata.name(c, "")
                if name.startswith("LATIN"):
                    lat += 1
                elif name.startswith("DEVANAGARI"):
                    dev += 1
    return lat / max(lat + dev, 1)


_r, _h = normalise("main kal office jaa raha hoon"), normalise("main kal ofis jaa hoon")
_s, _d, _i, _n = word_errors(_r, _h)
assert (_s, _d, _i, _n) == (1, 1, 0, 6), (_s, _d, _i, _n)          
assert normalise("मैं कल ऑफिस जा रहा हूँ।") == "मैं कल ऑफिस जा रहा हूँ", normalise("मैं कल ऑफिस जा रहा हूँ।")  # matras survive
assert normalise("Hello, World!!") == "hello world"
print("helper self-test passed")

helper self-test passed


## Step 3: Download the OpenSLR-104 test set and build the test sample
The test archive is about 443 MB (Hindi-English, 5.18 h, CC BY-SA 4.0). The OpenSLR page does **not** document the folder layout inside the archive, so we first print the folder tree, then use a loader that handles the usual Kaldi-style layout (`text`, `segments`, `wav.scp`) and also already-cut wav files.

**If `load_split` raises an error:** run the `show_tree` cell, copy the output and send it to your assistant. Do not guess.

In [ ]:
import os, random
from pathlib import Path
from math import gcd
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

def show_tree(root, max_depth=3, max_files=6):
    """Print the folder layout so you can SEE what is inside an extracted archive."""
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > max_depth:
            dirnames[:] = []
            continue
        print("  " * depth + (Path(dirpath).name or str(root)) + "/")
        for f in sorted(filenames)[:max_files]:
            print("  " * (depth + 1) + f)
        if len(filenames) > max_files:
            print("  " * (depth + 1) + f"... (+{len(filenames) - max_files} more files)")

def read_kv(path):
    """Read 'key value...' lines (Kaldi style) into a dict."""
    d = {}
    with open(path, encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.strip():
                k, _, v = line.partition(" ")
                d[k] = v.strip()
    return d

def load_split(root):
    """Find transcripts + audio under an extracted MUCS folder.
    Layout is not documented on the OpenSLR page, so two layouts are handled:
      (a) Kaldi style: text + segments (+ wav.scp)  -> audio is cut using start/end times
      (b) already-cut wav files named like the utterance ids in 'text'
    Returns a list of dicts: utt, path, start, end, dur, text."""
    root = Path(root)
    text_files = [p for p in root.rglob("text") if p.is_file()]
    if not text_files:
        raise FileNotFoundError(f"No file called 'text' under {root}. Run show_tree(root) and send me the output.")
    wavs = {p.stem: p for p in root.rglob("*.wav")}
    items, missing = [], 0
    for tf in text_files:
        d = tf.parent
        text = read_kv(tf)
        seg_path, scp_path = d / "segments", d / "wav.scp"
        if seg_path.exists():                                   
            scp = read_kv(scp_path) if scp_path.exists() else {}
            for line in open(seg_path, encoding="utf-8", errors="ignore"):
                parts = line.split()
                if len(parts) < 4 or parts[0] not in text:
                    continue
                utt, wid, s, e = parts[0], parts[1], float(parts[2]), float(parts[3])
                path = None
                for tok in scp.get(wid, "").split():            
                    if tok.endswith(".wav") and os.path.exists(tok):
                        path = Path(tok)
                        break
                path = path or wavs.get(wid) or wavs.get(Path(scp.get(wid, wid)).stem)
                if path is None:
                    missing += 1
                    continue
                items.append(dict(utt=utt, path=str(path), start=s, end=e, dur=e - s, text=text[utt]))
        else:                                                   
            for utt, txt in text.items():
                path = wavs.get(utt)
                if path is None:
                    missing += 1
                    continue
                items.append(dict(utt=utt, path=str(path), start=0.0, end=None, dur=sf.info(str(path)).duration, text=txt))
    print(f"found {len(items)} usable utterances ({missing} transcript lines had no audio)")
    return items

def load_audio(it, target_sr=16000):
    """Return a mono float32 array at 16 kHz for one item."""
    info = sf.info(it["path"])
    sr = info.samplerate
    if it["end"] is None:
        x, _ = sf.read(it["path"], dtype="float32")
    else:
        x, _ = sf.read(it["path"], start=int(it["start"] * sr), stop=int(it["end"] * sr), dtype="float32")
    if x.ndim > 1:
        x = x.mean(axis=1)
    if sr != target_sr:
        g = gcd(target_sr, sr)
        x = resample_poly(x, target_sr // g, sr // g).astype("float32")
    return x

def pick(items, n, seed, min_sec=1.0, max_sec=25.0):
    """Random subset of usable utterances (non-empty text, duration 1-25 s because Whisper takes 30 s windows)."""
    pool = [it for it in items if min_sec <= it["dur"] <= max_sec and normalise(it["text"])]
    random.Random(seed).shuffle(pool)
    print(f"pool={len(pool)} -> using {min(n, len(pool))} (~{sum(i['dur'] for i in pool[:n]) / 3600:.2f} h)")
    return pool[:n]

In [5]:
import subprocess, tarfile
from pathlib import Path

MIRRORS = ["https://www.openslr.org/resources/104/", "https://us.openslr.org/resources/104/", "https://openslr.elda.org/resources/104/"]

def download_and_extract(fname, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    tgz = out_dir / fname
    if not tgz.exists() or tgz.stat().st_size < 1_000_000:
        for base in MIRRORS:
            r = subprocess.run(["wget", "-q", base + fname, "-O", str(tgz)])
            if r.returncode == 0 and tgz.exists() and tgz.stat().st_size > 1_000_000:
                break
        else:
            raise RuntimeError("Download failed on every mirror. Download the file in your browser from https://openslr.org/104/ and upload it to " + str(out_dir))
    with tarfile.open(tgz) as t:
        t.extractall(out_dir)
    print("extracted", fname, "->", out_dir)

TEST_DIR = Path(WORK) / "data" / "test"
if not any(TEST_DIR.rglob("*.wav")):
    download_and_extract("Hindi-English_test.tar.gz", TEST_DIR)
show_tree(TEST_DIR, max_depth=3)

extracted Hindi-English_test.tar.gz -> /kaggle/working/pilot/data/test
test/
  Hindi-English_test.tar.gz
  test/
    406yMKxIdSDHRf8H.wav
    4oLp3bc9OSJbDrwM.wav
    4xyIm2P6Xzlin341.wav
    5baFeyjOxK8JQnNG.wav
    Bmcrju1ByfdrXE9P.wav
    FV8hkymGWcy53lrf.wav
    ... (+24 more files)
    transcripts/
      segments
      spk2utt
      spkr_list
      text
      utt2spk
      wav.scp


In [6]:
test_all = load_split(TEST_DIR)
test_items = pick(test_all, N_TEST, SEED)
test_audio = [load_audio(it) for it in test_items]
test_refs = [it["text"] for it in test_items]
print("example reference:", test_refs[0])

found 3136 usable utterances (0 transcript lines had no audio)
pool=3120 -> using 100 (~0.15 h)
example reference: यह आपको अगली slide पर ले जायेगा


## Step 4: Zero-shot baseline (the "before" number)
We run the pretrained Whisper-small, forced to transcribe in Hindi (`language="hi"`), with no training on our data.

**You should see:** a high WER (often 40-100% for the small model on this tutorial-style, mixed-script data). That is expected, and it is exactly why fine-tuning is needed. Also look at the "Latin share" lines: if the references and the model outputs write English words in different scripts, plain WER over-counts errors (this is the problem T-WER was designed for).

In [7]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model_zs = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE).eval()

@torch.no_grad()
def transcribe_arrays(model, arrays, batch_size=8):
    hyps = []
    for i in range(0, len(arrays), batch_size):
        feats = processor.feature_extractor(arrays[i:i + batch_size], sampling_rate=16000, return_tensors="pt").input_features.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            ids = model.generate(input_features=feats, language="hi", task="transcribe", max_new_tokens=100, no_repeat_ngram_size=4, repetition_penalty=1.2)
        hyps += processor.batch_decode(ids, skip_special_tokens=True)
    return [h.strip() for h in hyps]

t0 = time.time()
hyp_zs = transcribe_arrays(model_zs, test_audio)
df_zs, wer_zs = score_pairs(test_refs, hyp_zs)
print(f"ZERO-SHOT WER on {len(df_zs)} test utterances: {wer_zs:.1%}   ({time.time() - t0:.0f}s)")
print(f"Latin-script share  reference: {latin_share(df_zs.ref):.0%}   model output: {latin_share(df_zs.hyp):.0%}")
for _, r in df_zs.head(3).iterrows():
    print("\nREF:", r.ref, "\nHYP:", r.hyp)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers a

ZERO-SHOT WER on 100 test utterances: 85.1%   (32s)
Latin-script share  reference: 41%   model output: 7%

REF: यह आपको अगली slide पर ले जायेगा 
HYP: यहाँ आपको अगली स्लिट कर लेजायगा

REF: spoken tutorial project talktoateacher project का हिस्सा है 
HYP: श्पोकिन की तुयल परजेकत टाक्थ वो तीचर परहेंगा है

REF: ये समान क्रम में है जैसे कि वे window panel में दिखाई दे रहे थे 
HYP: अगरान क्रम्य हैं जेसीकि में बुंडो पैनलनी दिखाई देरही थी


## Step 5: Download the training set and fine-tune
The Hindi-English train archive is about **7.3 GB** (89.86 h). The download takes a few minutes. We then sample `N_TRAIN` utterances (not the whole set, because this is a pilot).

**What fine-tuning does:** Whisper turns audio into a log-Mel spectrogram, its encoder summarises it and its decoder predicts the transcript token by token. We keep training the same model on our Hindi-English data, minimising the cross-entropy between the predicted and true next token. We write the training loop by hand (about 40 lines) instead of using a trainer class, so it works across library versions and every part is visible.

**You should see:** the loss printed every 20 steps, falling from roughly 1-2 towards below 1. The run takes about 20-40 minutes on a T4 (my estimate, not measured). If you get *CUDA out of memory*, set `BATCH = 4` and `GRAD_ACCUM = 4`, or `FREEZE_ENCODER = True`, then re-run from this step.

In [8]:
TRAIN_DIR = Path(WORK) / "data" / "train"
if not any(TRAIN_DIR.rglob("*.wav")):
    download_and_extract("Hindi-English_train.tar.gz", TRAIN_DIR)
train_all = load_split(TRAIN_DIR)
train_items = pick(train_all, N_TRAIN, SEED)

extracted Hindi-English_train.tar.gz -> /kaggle/working/pilot/data/train
found 52825 usable utterances (0 transcript lines had no audio)
pool=52679 -> using 1500 (~2.49 h)


In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import get_linear_schedule_with_warmup

model_ft = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
processor.tokenizer.set_prefix_tokens(language="hindi", task="transcribe")   

class TrainSet(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]
        feats = processor.feature_extractor(load_audio(it), sampling_rate=16000).input_features[0]
        labels = processor.tokenizer(normalise(it["text"])).input_ids[:440]   
        return {"input_features": feats, "labels": labels}

def collate(batch):
    feats = torch.tensor(np.stack([b["input_features"] for b in batch]))
    L = max(len(b["labels"]) for b in batch)
    labels = torch.full((len(batch), L), -100, dtype=torch.long)            
    for i, b in enumerate(batch):
        labels[i, :len(b["labels"])] = torch.tensor(b["labels"])
    if (labels[:, 0] == START).all():                                       
        labels = labels[:, 1:]
    return {"input_features": feats, "labels": labels}

dl = DataLoader(TrainSet(train_items), batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2, drop_last=True)

model_ft.train(); model_ft.config.use_cache = False
model_ft.gradient_checkpointing_enable()                                    
if FREEZE_ENCODER:
    for p in model_ft.model.encoder.parameters(): p.requires_grad = False
params = [p for p in model_ft.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, WARMUP, MAX_STEPS)
use_amp = DEVICE == "cuda"
try:    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except Exception: scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

step = micro = 0; losses = []; t0 = time.time(); opt.zero_grad(set_to_none=True)
while step < MAX_STEPS:
    for batch in dl:
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = model_ft(input_features=batch["input_features"].to(DEVICE), labels=batch["labels"].to(DEVICE)).loss / GRAD_ACCUM
        scaler.scale(loss).backward(); micro += 1
        if micro % GRAD_ACCUM == 0:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
            losses.append(loss.item() * GRAD_ACCUM)
            if step % 20 == 0:
                print(f"step {step}/{MAX_STEPS}  loss {np.mean(losses[-20:]):.3f}  elapsed {(time.time() - t0) / 60:.1f} min")
            if step >= MAX_STEPS: break
model_ft.eval(); model_ft.config.use_cache = True; model_ft.gradient_checkpointing_disable()
print("training done")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

step 20/400  loss 1.039  elapsed 0.9 min
step 40/400  loss 0.719  elapsed 1.7 min
step 60/400  loss 0.630  elapsed 2.6 min
step 80/400  loss 0.525  elapsed 3.5 min
step 100/400  loss 0.470  elapsed 4.4 min
step 120/400  loss 0.294  elapsed 5.2 min
step 140/400  loss 0.325  elapsed 6.1 min
step 160/400  loss 0.364  elapsed 6.9 min
step 180/400  loss 0.328  elapsed 7.8 min
step 200/400  loss 0.233  elapsed 8.6 min
step 220/400  loss 0.209  elapsed 9.5 min
step 240/400  loss 0.274  elapsed 10.3 min
step 260/400  loss 0.236  elapsed 11.2 min
step 280/400  loss 0.199  elapsed 12.0 min
step 300/400  loss 0.131  elapsed 12.9 min
step 320/400  loss 0.123  elapsed 13.7 min
step 340/400  loss 0.135  elapsed 14.6 min
step 360/400  loss 0.145  elapsed 15.4 min
step 380/400  loss 0.158  elapsed 16.3 min
step 400/400  loss 0.118  elapsed 17.2 min
training done


## Step 6: Fine-tuned model (the "after" number) and results table
We score the fine-tuned model on the **same** 100 test utterances, then save a results file you can copy numbers from.

In [10]:
hyp_ft = transcribe_arrays(model_ft, test_audio)
df_ft, wer_ft = score_pairs(test_refs, hyp_ft)
print(f"BEFORE (zero-shot): {wer_zs:.1%}   AFTER ({MAX_STEPS} steps on {len(train_items)} utterances): {wer_ft:.1%}")

per_utt = df_zs[["ref", "ref_words", "errors"]].rename(columns={"errors": "errors_zero_shot"}).copy()
per_utt["hyp_zero_shot"] = df_zs.hyp.values
per_utt["hyp_finetuned"] = df_ft.hyp.values
per_utt["errors_finetuned"] = df_ft.errors.values
per_utt.to_csv(f"{WORK}/per_utterance_results.csv", index=False)
results = dict(model=MODEL_NAME, n_test=len(df_zs), n_train=len(train_items), steps=MAX_STEPS,
               wer_zero_shot=wer_zs, wer_finetuned=wer_ft, final_loss=float(np.mean(losses[-20:])))
json.dump(results, open(f"{WORK}/pilot_results.json", "w"), indent=2)
display(pd.DataFrame([results]).T.rename(columns={0: "value"}))
for _, r in per_utt.head(4).iterrows():
    print("\nREF :", r.ref, "\nZERO:", r.hyp_zero_shot, "\nFT  :", r.hyp_finetuned)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

BEFORE (zero-shot): 85.1%   AFTER (400 steps on 1500 utterances): 48.4%


,value
model,openai/whisper-small
n_test,100
n_train,1500
steps,400
wer_zero_shot,0.850543
wer_finetuned,0.483696
final_loss,0.117862



REF : यह आपको अगली slide पर ले जायेगा 
ZERO: यहाँ आपको अगली स्लिट कर लेजायगा 
FT  : यह आपको अगली slide पर ले जाएँगा

REF : spoken tutorial project talktoateacher project का हिस्सा है 
ZERO: श्पोकिन की तुयल परजेकत टाक्थ वो तीचर परहेंगा है 
FT  : spoken tutorial project talktoa teacher project का हिस्था है

REF : ये समान क्रम में है जैसे कि वे window panel में दिखाई दे रहे थे 
ZERO: अगरान क्रम्य हैं जेसीकि में बुंडो पैनलनी दिखाई देरही थी 
FT  : अतलान क्रम में है चैसे कि बें विंडो पैनल मी दिखाई दे रहे थी

REF : highlight current line option को checkmark करें close button पर click करें 
ZERO: अदर्याने किली 
FT  : current line option को checkmark करें close button पर click करेम


## Step 7: A slice of the VAANI Benchmark, split by group
**First, one-time setup (do this before the lecture day):** on the Hugging Face website, open the dataset page `ARTPARK-IISc/Vaani-Benchmark-V1.0` while logged in and accept its terms. Then create a *read* token in your Hugging Face settings. If you skip the terms, loading fails with an access error.

**This step is "inspect, then fill in".** I could not look inside the dataset, so the column names are unknown. The first cell prints the columns. Then you set `AUDIO_COL`, `REF_COLS` (the three human transcripts), `GROUP_COL` (for example gender or state) and `SPEAKER_COL` by looking at that printout.

**Important:** the official VAANI toolkit treats missing predictions as empty, so it only gives meaningful scores when you run **all** 5,050 test rows. For the pilot we therefore use our own minimum-error multi-reference score on a random slice. The final numbers must come from the official toolkit on the full set.

In [11]:
from huggingface_hub import notebook_login
notebook_login()
from datasets import load_dataset
VAANI = "ARTPARK-IISc/Vaani-Benchmark-V1.0"
ds = load_dataset(VAANI, split="test", streaming=True)
ex = next(iter(ds))
for k, v in ex.items():
    if isinstance(v, dict) and ("array" in v or "bytes" in v):
        print(f"{k:25s} -> AUDIO")
    else:
        print(f"{k:25s} -> {str(v)[:90]!r}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

audio                     -> AUDIO
speakerID                 -> '100698'
language                  -> 'Hindi'
state                     -> 'Chhattisgarh'
district                  -> 'Sarguja'
duration                  -> '11.448'
gender                    -> 'male'
reference_image           -> 'Images/IISc_VaaniProject_Sarguja-SPECIFIC_01274.jpg'
transcription_v1          -> '<noise> ठते {बैठते} है, जय माला के टाइम {time} [breathing] ओ हि वाला कुर्सी है, <PAUSE> और'
transcription_v2          -> '<noise> ते है, जय माला के टाइम {time} [breathing] ओ हि वाला कुर्सी है, <PAUSE> और रखा हुआ '
transcription_v3          -> '<noise> ते हैं, जयमाला के टाइप {type} [breathing] वो ही वाला कुर्सी है, <PAUSE> और रखा हुआ'


In [15]:
AUDIO_COL   = "audio"
REF_COLS    = ["transcription_v1", "transcription_v2", "transcription_v3"]
GROUP_COL   = "state"
SPEAKER_COL = "speakerID"

In [16]:
from datasets import Audio
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd

def to_array(a, target=16000):
    """Decode a Hugging Face audio dict WITHOUT torchcodec (decode=False gives raw bytes)."""
    x, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32") if a.get("bytes") else sf.read(a["path"], dtype="float32")
    if x.ndim > 1: x = x.mean(axis=1)
    if sr != target:
        g = gcd(target, sr); x = resample_poly(x, target // g, sr // g).astype("float32")
    return x

assert GROUP_COL is None or GROUP_COL in ex, "GROUP_COL not found - check the printout above"
ds2 = ds.cast_column(AUDIO_COL, Audio(decode=False))
sample = list(ds2.shuffle(seed=SEED, buffer_size=2000).take(N_VAANI))
v_audio = [to_array(s[AUDIO_COL]) for s in sample]
print(len(sample), "VAANI utterances loaded; example group values:", [s.get(GROUP_COL) for s in sample[:5]] if GROUP_COL else "n/a")

200 VAANI utterances loaded; example group values: ['Odisha', 'Punjab', 'Haryana', 'MadhyaPradesh', 'MadhyaPradesh']


In [17]:
def vaani_table(model, name):
    hyps = transcribe_arrays(model, v_audio)
    rows = []
    for s, h, a in zip(sample, hyps, v_audio):
        best = multiref_errors([s[c] for c in REF_COLS], h)
        if best is None: continue
        rows.append(dict(errors=best[0], ref_words=best[1], duration=len(a) / 16000,
                         group=s.get(GROUP_COL) if GROUP_COL else None,
                         speaker=s.get(SPEAKER_COL) if SPEAKER_COL else None))
    df = pd.DataFrame(rows)
    print(f"{name}: pilot multi-reference WER on {len(df)} VAANI utterances = {df.errors.sum() / df.ref_words.sum():.1%}")
    return df

vaani_zs = vaani_table(model_zs, "zero-shot")
vaani_ft = vaani_table(model_ft, "fine-tuned")
vaani_ft.to_csv(f"{WORK}/vaani_pilot_finetuned.csv", index=False)
if GROUP_COL:
    print(vaani_ft.groupby("group").agg(utts=("errors", "size"), errors=("errors", "sum"), words=("ref_words", "sum")).assign(WER=lambda d: d.errors / d.words))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

zero-shot: pilot multi-reference WER on 200 VAANI utterances = 74.0%


[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

fine-tuned: pilot multi-reference WER on 200 VAANI utterances = 60.4%
                  utts  errors  words       WER
group                                          
ArunachalPradesh    14     114    185  0.616216
Assam               30     175    374  0.467914
Delhi                5      86    124  0.693548
Gujarat             33     312    503  0.620278
Haryana             27     224    361  0.620499
MadhyaPradesh       34     227    437  0.519451
Odisha              27     357    505  0.706931
Punjab              20     168    284  0.591549
Tripura             10      87    122  0.713115


## Step 8: Bias analysis: Poisson drop-in-deviance test and bootstrap intervals
This is the statistical core of the project, in miniature.

1. **Poisson regression with an offset:** word errors per utterance are counts, and `log(ref_words)` is the offset, so the model describes errors **per word** (WER).
2. **Drop-in-deviance test:** fit a model without the group and one with it. The drop in deviance `D` is compared with a chi-squared distribution. A small p-value means the group explains a significant part of the errors. The **rate ratio** is the size of the gap (1.20 = that group's WER is 20% higher).
3. **Two p-values:** the *naive* one assumes every utterance is independent; the *adjusted* one divides D by the dispersion. When one speaker contributes many utterances the naive p is too optimistic, which is why the real project uses a speaker random effect.
4. **Bootstrap:** resample whole speakers to get a 95% interval for each group's WER. A wide interval means a small, uncertain group.

First we test the code on the **hand example from the study guide** (150/1000 vs 200/1000 errors, expected D = 7.17, p about 0.007), then on **synthetic** data with a planted effect, then on your VAANI slice if you set `GROUP_COL`.

In [18]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

def drop_in_deviance(df, group, controls=()):
    """Drop-in-deviance (likelihood-ratio) test for ONE grouping variable.
    df needs columns: errors, ref_words, <group>, and any control columns.
    Poisson regression with offset = log(ref_words), so the model describes errors PER WORD (= WER).
    Returns a dict with D, degrees of freedom, naive p-value, dispersion-adjusted p-value and rate ratios."""
    df = df.dropna(subset=[group, "errors", "ref_words"] + list(controls)).copy()
    base = " + ".join(controls) if controls else "1"
    f0 = f"errors ~ {base}"
    f1 = f"errors ~ C({group}) + {base}" if controls else f"errors ~ C({group})"
    kw = dict(data=df, family=sm.families.Poisson(), offset=np.log(df["ref_words"]))
    m0 = smf.glm(f0, **kw).fit()
    m1 = smf.glm(f1, **kw).fit()
    D = m0.deviance - m1.deviance                      # the "drop in deviance"
    k = int(round(m1.df_model - m0.df_model))          # number of extra parameters = groups - 1
    p_naive = float(stats.chi2.sf(D, k))
    phi = max(m1.pearson_chi2 / m1.df_resid, 1.0)      # >1 means errors are more spread out than Poisson allows
    p_adj = float(stats.chi2.sf(D / phi, k))           # quasi-Poisson style correction
    rr = np.exp(m1.params).drop("Intercept")
    ci = np.exp(m1.conf_int()).drop("Intercept")
    table = pd.DataFrame({"rate_ratio": rr, "ci_low": ci[0], "ci_high": ci[1]})
    return dict(D=float(D), df=k, p_naive=p_naive, dispersion=float(phi), p_adjusted=p_adj,
                null_deviance=float(m0.deviance), full_deviance=float(m1.deviance), rate_ratios=table)

def bootstrap_wer_ci(df, group, speaker, B=1000, seed=0, level=95):
    """Bootstrap CI of group WER, resampling WHOLE SPEAKERS (utterances of one speaker are correlated)."""
    rng = np.random.default_rng(seed)
    out = []
    for g, sub in df.groupby(group):
        per_spk = sub.groupby(speaker)[["errors", "ref_words"]].sum()
        e, w = per_spk["errors"].to_numpy(), per_spk["ref_words"].to_numpy()
        n = len(e)
        wers = np.empty(B)
        for b in range(B):
            idx = rng.integers(0, n, n)
            wers[b] = e[idx].sum() / w[idx].sum()
        lo, hi = np.percentile(wers, [(100 - level) / 2, 100 - (100 - level) / 2])
        out.append(dict(group=g, speakers=n, wer=e.sum() / w.sum(), ci_low=lo, ci_high=hi))
    return pd.DataFrame(out)

def make_synthetic(n_speakers=60, utts=8, rate_ratio=1.3, base_wer=0.25, speaker_sd=0.25, seed=1):
    """SYNTHETIC data to rehearse the method (planted group effect + speaker effects). Not real results."""
    rng = np.random.default_rng(seed)
    rows = []
    for s in range(n_speakers):
        grp = "B" if s % 2 else "A"
        u = rng.normal(0, speaker_sd)
        for _ in range(utts):
            n = int(rng.integers(5, 30))
            lam = n * base_wer * (rate_ratio if grp == "B" else 1.0) * np.exp(u)
            rows.append(dict(speaker=f"spk{s}", group=grp, ref_words=n, errors=int(rng.poisson(lam)),
                             duration=n * 0.45 + rng.normal(0, 0.3)))
    return pd.DataFrame(rows)

In [19]:
# 1) Hand example from the study guide: expect D = 7.17, p = 0.0074, rate ratio = 1.33
hand = pd.DataFrame({"group": ["A", "B"], "errors": [150, 200], "ref_words": [1000, 1000]})
r = drop_in_deviance(hand, "group")
print(f"hand example: D = {r['D']:.2f}, df = {r['df']}, p = {r['p_naive']:.4f}, rate ratio = {r['rate_ratios'].rate_ratio.iloc[0]:.2f}")
assert abs(r["D"] - 7.17) < 0.01

# 2) Synthetic data (NOT real results): group B has a planted 30% higher error rate, plus speaker-to-speaker variation
syn = make_synthetic(rate_ratio=1.3)
res = drop_in_deviance(syn, "group", controls=("duration",))
print(f"\nSYNTHETIC  D = {res['D']:.1f}, df = {res['df']}, naive p = {res['p_naive']:.2g}, adjusted p = {res['p_adjusted']:.2g}, dispersion = {res['dispersion']:.2f}")
display(res["rate_ratios"].round(3))
display(bootstrap_wer_ci(syn, "group", "speaker", B=500).round(3))

hand example: D = 7.17, df = 1, p = 0.0074, rate ratio = 1.33

SYNTHETIC  D = 55.4, df = 1, naive p = 1e-13, adjusted p = 5.7e-12, dispersion = 1.17


/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(


,rate_ratio,ci_low,ci_high
C(group)[T.B],1.376,1.265,1.498
duration,1.009,0.995,1.024


,group,speakers,wer,ci_low,ci_high
0,A,30,0.223,0.203,0.245
1,B,30,0.307,0.284,0.334


In [20]:
# 3) Your VAANI pilot slice (runs only if GROUP_COL is set and there are at least 2 groups)
if GROUP_COL and vaani_ft["group"].nunique() >= 2:
    d = vaani_ft.dropna(subset=["group"]).copy()
    d["group"] = d["group"].astype(str)
    big = d["group"].value_counts(); d = d[d["group"].isin(big[big >= 5].index)]     # drop groups with fewer than 5 utterances
    res = drop_in_deviance(d, "group", controls=("duration",))
    print(f"VAANI pilot, groups = {sorted(d.group.unique())}")
    print(f"D = {res['D']:.2f}, df = {res['df']}, naive p = {res['p_naive']:.3g}, adjusted p = {res['p_adjusted']:.3g}")
    display(res["rate_ratios"].round(3))
    if d["speaker"].notna().any():
        display(bootstrap_wer_ci(d, "group", "speaker", B=1000).round(3))
    print("Reminder: this is a small pilot slice. Report it as 'method demonstration', not as evidence of bias.")
else:
    print("GROUP_COL not set (or only one group): use the synthetic demo above to explain the method.")

VAANI pilot, groups = ['ArunachalPradesh', 'Assam', 'Delhi', 'Gujarat', 'Haryana', 'MadhyaPradesh', 'Odisha', 'Punjab', 'Tripura']
D = 29.12, df = 8, naive p = 0.000303, adjusted p = 0.000303


,rate_ratio,ci_low,ci_high
C(group)[T.Assam],0.745,0.589,0.944
C(group)[T.Delhi],1.021,0.767,1.359
C(group)[T.Gujarat],0.962,0.775,1.195
C(group)[T.Haryana],0.954,0.760,1.198
C(group)[T.MadhyaPradesh],0.790,0.629,0.993
C(group)[T.Odisha],1.071,0.863,1.327
C(group)[T.Punjab],0.828,0.640,1.070
C(group)[T.Tripura],1.113,0.841,1.473
duration,1.033,1.013,1.054


,group,speakers,wer,ci_low,ci_high
0,ArunachalPradesh,11,0.616,0.507,0.701
1,Assam,23,0.468,0.405,0.530
2,Delhi,5,0.694,0.416,0.876
3,Gujarat,21,0.620,0.561,0.684
4,Haryana,21,0.620,0.550,0.685
5,MadhyaPradesh,31,0.519,0.456,0.582
6,Odisha,21,0.707,0.646,0.763
7,Punjab,19,0.592,0.495,0.665
8,Tripura,5,0.713,0.573,0.842


Reminder: this is a small pilot slice. Report it as 'method demonstration', not as evidence of bias.


## Step 9: What to say and show
- **Slide 1 (what we built):** a pipeline that downloads OpenSLR-104, scores Whisper before and after fine-tuning, and runs a Poisson drop-in-deviance test with bootstrap intervals.
- **Slide 2 (pilot numbers):** `pilot_results.json` (zero-shot WER, fine-tuned WER, n_test, n_train, steps), labelled *pilot: Whisper-small, small subset*.
- **Slide 3 (method check):** the hand example (D = 7.17) and the synthetic example show the statistics code works.
- **Slide 4 (what comes next):** Whisper Large-v3 with LoRA on an A100, the full OpenSLR-104 training set, all 5,050 VAANI rows scored with the official toolkit, speaker random effects (R lme4 or glmmTMB), and district/state/gender splits.

**Honest limitations to state:** Whisper-small, a few hundred steps, 100 test utterances, plain WER (script mismatch inflates it), a minimum-error multi-reference score that is only a stand-in for the official toolkit, and a VAANI slice too small to claim anything about bias.

**Files to download from the work folder:** `pilot_results.json`, `per_utterance_results.csv`, `vaani_pilot_finetuned.csv`.

In [21]:
import os
import zipfile
from google.colab import files

files_to_download = [
    "pilot_results.json",
    "per_utterance_results.csv",
    "vaani_pilot_finetuned.csv"
]

zip_filename = "pilot_results_v2.zip"
zip_filepath = f"{WORK}/{zip_filename}"

# Bundle the files into a single zip
print(f"Creating {zip_filename}...")
with zipfile.ZipFile(zip_filepath, 'w') as zipf:
    for filename in files_to_download:
        filepath = f"{WORK}/{filename}"
        if os.path.exists(filepath):
            zipf.write(filepath, arcname=filename)
            print(f"  Added {filename}")
        else:
            print(f"  Skipped {filename} (not found)")

# Trigger a single download
print("\nDownloading zip file...")
files.download(zip_filepath)

Creating pilot_results_v2.zip...
  Added pilot_results.json
  Added per_utterance_results.csv
  Added vaani_pilot_finetuned.csv



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>